In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import splink

%matplotlib inline

os.chdir(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa')

rraa_extranjeros = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\raw\hub_personas.parquet')
censo = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\raw\censo.parquet')

In [2]:
censo_extranjeros = censo[censo["id_pais_documento"] != "858"]

rraa_extranjeros["id_pais_documento"] = rraa_extranjeros["id_pais_documento"].replace("nan", pd.NA).fillna(0).astype(float).astype(int).astype(str)

In [4]:
vinculados = pd.merge(censo_extranjeros, rraa_extranjeros, on = ["id_pais_documento", "documento"], how = "left")

prob :)

In [3]:
df_l = censo_extranjeros[["id_censo", "id_pais_documento", "documento", "primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido", "fecha_nacimiento"]].rename(columns = {"id_censo":"unique_id"})

df_r = rraa_extranjeros[["id_estadistico_persona", "id_pais_documento", "documento", "primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido", "fecha_nacimiento"]]

df_r["id_estadistico_persona"] = df_r["id_estadistico_persona"].astype(str).str.split('.').str[0]

df_r = df_r.drop_duplicates(
    subset = ["id_estadistico_persona", "id_pais_documento", "documento"]).sort_values(
        by = ["id_estadistico_persona", "id_pais_documento", "documento"])

df_r["row"] = df_r.groupby("id_estadistico_persona").cumcount() + 1

df_r["unique_id"] = df_r["id_estadistico_persona"] + "_" + df_r["row"].astype(str)

df_r = df_r.reindex(columns=["unique_id"] + [col for col in df_r.columns if col != "unique_id"])

df_r = df_r.drop(columns=["row", "id_estadistico_persona"])

C:\Users\lpescetto\AppData\Local\Temp\ipykernel_22120\1132308558.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_r["id_estadistico_persona"] = df_r["id_estadistico_persona"].astype(str).str.split('.').str[0]


In [39]:
vinculados = pd.merge(df_l, df_r, on = ["id_pais_documento", "documento"], how = "left")

In [4]:
df_l["primer_nombre"] = df_l["primer_nombre"].replace("", np.nan)
df_l["segundo_nombre"] = df_l["segundo_nombre"].replace("", np.nan)
df_l["primer_apellido"] = df_l["primer_apellido"].replace("", np.nan)
df_l["segundo_apellido"] = df_l["segundo_apellido"].replace("", np.nan)

df_r["primer_nombre"] = df_r["primer_nombre"].replace("", np.nan)
df_r["segundo_nombre"] = df_r["segundo_nombre"].replace("", np.nan)
df_r["primer_apellido"] = df_r["primer_apellido"].replace("", np.nan)
df_r["segundo_apellido"] = df_r["segundo_apellido"].replace("", np.nan)

df_l["fecha_nacimiento"] = df_l["fecha_nacimiento"].dt.strftime('%Y-%m-%d')
df_r["fecha_nacimiento"] = df_r["fecha_nacimiento"].dt.strftime('%Y-%m-%d')

df_l["unique_id"] = df_l["unique_id"].astype(object)
df_l["id_pais_documento"] = df_l["id_pais_documento"].astype(object)

In [5]:
# se chequea que los df cumplan los requisitos para implementar el modelo

# 1. id unico en cada tabla
print("1:")
print("id_unico en censo:", len(df_l) == len(df_l.drop_duplicates(subset = "unique_id")))
print("id_unico en rraa:", len(df_r) == len(df_r.drop_duplicates(subset = "unique_id")))
print("\n" * 1)

# 2. los tipos de datos deben ser iguales
print("2:")
print(df_l.dtypes == df_r.dtypes)

# 3. se recuerda que los nulls son tratados de forma distinto que las strings vacías (en los nombres hay strings vacías y en el resto de las variables nulls)

1:
id_unico en censo: True
id_unico en rraa: True


2:
unique_id            True
id_pais_documento    True
documento            True
primer_nombre        True
segundo_nombre       True
primer_apellido      True
segundo_apellido     True
fecha_nacimiento     True
dtype: bool


In [6]:
from splink.duckdb.linker import DuckDBLinker

linker_censo_extranjeros = DuckDBLinker(df_l)
linker_rraa_extranjeros = DuckDBLinker(df_r)

In [8]:
linker_censo_extranjeros.missingness_chart()

alt.LayerChart(...)

In [9]:
linker_rraa_extranjeros.missingness_chart()

alt.LayerChart(...)

In [7]:
documentos_l = df_l.groupby(["documento"]).size().reset_index(name='count').sort_values(by = 'count', ascending=False)

documentos_r = df_r.groupby(["documento"]).size().reset_index(name='count').sort_values(by = 'count', ascending=False)

In [8]:
documentos_genericos_l = ["0", "00", "000", "00000000", "no sabe", "no recuerdo", "no lo recuerdo", "no lo se", "no se acuerda", "no recuerda", "no"]

documentos_genericos_r = ["p", "00000000", "1", "2", "3", "11", "8", "111", "6", "37", "4", "oo", "aconf", "1111", "1111111111", "11111111", "111111"]

In [9]:
df_l.loc[df_l["documento"].isin(documentos_genericos_l), "documento"] = np.nan

df_r.loc[df_r["documento"].isin(documentos_genericos_r), "documento"] = np.nan

In [12]:
linker_censo_extranjeros = DuckDBLinker(df_l)
linker_censo_extranjeros.missingness_chart()

alt.LayerChart(...)

In [13]:
linker_rraa_extranjeros.profile_columns()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.VConcatChart(...)

## Reglas de bloqueo y comparaciones

In [10]:
from splink.duckdb.blocking_rule_library import block_on

settings = {"link_type": "link_only"}

linker_br = DuckDBLinker([df_r, df_l], settings)

# br0 = block_on(["id_pais_documento"]) (demoró 15 minutos así que es demasiado)
br1 = block_on(["id_pais_documento", "documento"])
br2 = block_on(["id_pais_documento", "primer_apellido"])
# br3 = block_on(["id_pais_documento", "primer_nombre"])
br4 = block_on(["id_pais_documento", "fecha_nacimiento"])
br5 = block_on(["primer_nombre", "primer_apellido", "fecha_nacimiento"])
br6 = block_on(["id_pais_documento", "primer_nombre", "substr(primer_apellido, 1,1)"])
br7 = block_on(["id_pais_documento", "primer_nombre", "substr(fecha_nacimiento, 1,4)"])

list_br = [br1, br2, br4, br5, br6, br7]

# for br in list_br:
#     count = linker_br.count_num_comparisons_from_blocking_rule(br)
#     print(f"Comparisons generated by '{br.blocking_rule_sql}': {count:,.0f}")

In [31]:
linker_br.cumulative_num_comparisons_from_blocking_rules_chart(list_br)

alt.Chart(...)

In [11]:
from scripts.comparaciones import get_comparacion_fecha_nacimiento, get_comparacion_nombres_combinados, get_comparacion_nombre
import splink.duckdb.comparison_library as cl
import splink.duckdb.comparison_template_library as ctl

comparacion_fecha_nacimiento = get_comparacion_fecha_nacimiento(con_dc = False)
#comparacion_nombres = get_comparacion_nombres_combinados("primer_nombre", "segundo_nombre", "nombres")
#comparacion_apellidos = get_comparacion_nombres_combinados("primer_apellido", "segundo_apellido", "apellidos")
comparacion_documento = cl.levenshtein_at_thresholds("documento", [1])
comparacion_id_pais_documento = cl.exact_match("id_pais_documento")
comparacion_id_tipo_documento = cl.exact_match("id_tipo_documento")

comparacion_primer_nombre = get_comparacion_nombre("primer_nombre", usar_tfa = True)
comparacion_segundo_nombre = get_comparacion_nombre("segundo_nombre", usar_tfa = True)
comparacion_primer_apellido = get_comparacion_nombre("primer_apellido", usar_tfa = True)
comparacion_segundo_apellido = get_comparacion_nombre("segundo_apellido", usar_tfa = True)

In [12]:
settings = {
    "link_type": "link_only",
    "comparisons": [
        comparacion_fecha_nacimiento,
        comparacion_primer_nombre,
        comparacion_segundo_nombre,
        comparacion_primer_apellido,
        comparacion_segundo_apellido,
        comparacion_documento,
        #comparacion_id_pais_documento
    ],
    "blocking_rules_to_generate_predictions": [
        br1, br2, br5, br6, br7
    ],
    "em_convergence": 1e-8,
    "max_iterations": 25,
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = DuckDBLinker([df_l, df_r], settings)

In [64]:
deterministic_rules = [
    "l.primer_nombre = r.primer_nombre",
    "l.primer_apellido = r.primer_apellido",
    "l.fecha_nacimiento = r.fecha_nacimiento",
    "l.documento = r.documento",
    "l.id_pais_documento = r.id_pais_documento"
]

linker.estimate_probability_two_random_records_match(deterministic_rules, recall = 0.9)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
linker.estimate_u_using_random_sampling(max_pairs=1e8)

----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - fecha_nacimiento (no m values are trained).
    - primer_nombre (no m values are trained).
    - segundo_nombre (no m values are trained).
    - primer_apellido (no m values are trained).
    - segundo_apellido (no m values are trained).
    - documento (no m values are trained).


In [16]:
t_br1 = block_on(["documento", "id_pais_documento", "fecha_nacimiento"])
training_session1 = linker.estimate_parameters_using_expectation_maximisation(t_br1)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."documento" = r."documento") AND (l."id_pais_documento" = r."id_pais_documento") AND (l."fecha_nacimiento" = r."fecha_nacimiento")

Parameter estimates will be made for the following comparison(s):
    - primer_nombre
    - segundo_nombre
    - primer_apellido
    - segundo_apellido

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - fecha_nacimiento
    - documento

Iteration 1: Largest change in params was -0.0985 in the m_probability of primer_apellido, level `Exact match primer_apellido`
Iteration 2: Largest change in params was -0.00181 in the m_probability of segundo_nombre, level `All other comparisons`
Iteration 3: Largest change in params was -0.000474 in probability_two_random_records_match
Iteration 4: Largest change in params was -0.000692 in probability_two_random_records_match
Iteration 5: Largest chang

In [14]:
t_br2 = block_on(["primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido", "fecha_nacimiento"])
training_session2 = linker.estimate_parameters_using_expectation_maximisation(t_br2)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."primer_nombre" = r."primer_nombre") AND (l."segundo_nombre" = r."segundo_nombre") AND (l."primer_apellido" = r."primer_apellido") AND (l."segundo_apellido" = r."segundo_apellido") AND (l."fecha_nacimiento" = r."fecha_nacimiento")

Parameter estimates will be made for the following comparison(s):
    - documento

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - fecha_nacimiento
    - primer_nombre
    - segundo_nombre
    - primer_apellido
    - segundo_apellido

Iteration 1: Largest change in params was -0.363 in the m_probability of documento, level `Exact match`
Iteration 2: Largest change in params was 2.22e-16 in the m_probability of documento, level `Exact match`

EM converged after 2 iterations

Your model is not yet fully trained. Missing estimates for:
    - fecha_nacimiento (no m values are trained).
    

In [17]:
t_br3 = block_on(["primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido", "documento", "id_pais_documento"])
training_session3 = linker.estimate_parameters_using_expectation_maximisation(t_br3)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."primer_nombre" = r."primer_nombre") AND (l."segundo_nombre" = r."segundo_nombre") AND (l."primer_apellido" = r."primer_apellido") AND (l."segundo_apellido" = r."segundo_apellido") AND (l."documento" = r."documento") AND (l."id_pais_documento" = r."id_pais_documento")

Parameter estimates will be made for the following comparison(s):
    - fecha_nacimiento

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - primer_nombre
    - segundo_nombre
    - primer_apellido
    - segundo_apellido
    - documento

Level All other comparisons on comparison fecha_nacimiento not observed in dataset, unable to train m value

Iteration 1: Largest change in params was 6.54e-14 in probability_two_random_records_match

EM converged after 1 iterations
m probability not trained for fecha_nacimiento - All other comparisons (comparison vect

In [18]:
linker.match_weights_chart()

alt.VConcatChart(...)

In [19]:
predictions = linker.predict(threshold_match_probability=0.2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'fecha_nacimiento':
    m values not fully trained
The 'probability_two_random_records_match' setting has been set to the default value (0.0001). 
If this is not the desired behaviour, either: 
 - assign a value for `probability_two_random_records_match` in your settings dictionary, or 
 - estimate with the `linker.estimate_probability_two_random_records_match` function.


In [20]:
df_predictions = predictions.as_pandas_dataframe()

In [21]:
max_match_probabilities = df_predictions.groupby('unique_id_l')['match_weight'].idxmax()

predictions_max_match_prob = df_predictions.loc[max_match_probabilities]

In [22]:
records_to_view  = predictions_max_match_prob.sample(n=50, random_state = 13).to_dict(orient="records")
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

In [23]:
predictions_max_match_prob.value_counts("gamma_documento")

gamma_documento
 2    3953
 0    1379
 1     228
-1     151
Name: count, dtype: int64

In [24]:
a = predictions_max_match_prob[predictions_max_match_prob["gamma_documento"] == 2]

In [28]:
predictions_max_match_prob.value_counts("match_key")

match_key
0    3931
1    1297
4     271
3     142
2      70
Name: count, dtype: int64

In [30]:
b = predictions_max_match_prob[predictions_max_match_prob["match_key"] == "2"]

In [39]:
corte1 = predictions_max_match_prob[predictions_max_match_prob["match_probability"] > 0.99]

corte1.value_counts("gamma_documento", normalize=True)

gamma_documento
 2    0.797735
 0    0.138147
 1    0.045712
-1    0.018406
Name: proportion, dtype: float64

In [40]:
corte2 = predictions_max_match_prob[predictions_max_match_prob["match_probability"] > 0.999]

corte2.value_counts("gamma_documento", normalize=True)

gamma_documento
 2    0.791464
 0    0.138019
 1    0.052192
-1    0.018325
Name: proportion, dtype: float64

In [41]:
corte3 = predictions_max_match_prob[predictions_max_match_prob["match_probability"] > 0.9]

corte3.value_counts("gamma_documento", normalize=True)

gamma_documento
 2    0.749858
 0    0.185045
 1    0.043082
-1    0.022016
Name: proportion, dtype: float64

In [33]:
records_to_view  = corte1.sample(n=50, random_state = 13).to_dict(orient="records")
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

In [44]:
a = corte1[corte1["gamma_documento"] != 2]

In [48]:
corte1_ajustado = corte1[(corte1["gamma_documento"] == 2) | ((corte1["gamma_fecha_nacimiento"] == 2) & (corte1["gamma_primer_nombre"] == 2) & (corte1["gamma_primer_apellido"] == 2))]

In [46]:
a = corte1_ajustado[corte1_ajustado["gamma_documento"] != 2]

In [51]:
extranjeros_vinculados = corte1_ajustado[["unique_id_l", "unique_id_r"]]

In [52]:
from utils.saveFile import guardar_como_parquet

guardar_como_parquet(extranjeros_vinculados, r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\extranjeros_vinculados.parquet')

Se guardó el DataFrame como Parquet en: C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\extranjeros_vinculados.parquet
